# Inverse Rollout: Optimization Solver

This notebook demonstrates how to use the `InverseRolloutSolver` to find initial condition perturbations that produce a specific desired change in the model output trajectory.

**Use case**: You have a reference forecast and want to find the smallest perturbation to the initial conditions that would produce a specific change in the output (e.g., making the temperature 5K lower at a specific location over several forecast steps).

This is an iterative optimization approach that minimizes the difference between the achieved trajectory change and your target trajectory change.

## Prerequisites

This notebook requires the same data as the ERA5 example. Please run `example_era5.ipynb` first to download the required data.

```
pip install cdsapi matplotlib
```

## Load the Data

In [ ]:
from pathlib import Path

import torch
import xarray as xr

from aurora import Batch, Metadata

# Path where ERA5 data was downloaded
download_path = Path("~/downloads").expanduser()

# Load the datasets
static_vars_ds = xr.open_dataset(download_path / "static.nc", engine="netcdf4")
surf_vars_ds = xr.open_dataset(download_path / "2023-01-01-surface-level.nc", engine="netcdf4")
atmos_vars_ds = xr.open_dataset(download_path / "2023-01-01-atmospheric.nc", engine="netcdf4")

# Create the batch
batch = Batch(
    surf_vars={
        "2t": torch.from_numpy(surf_vars_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_vars_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_vars_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_vars_ds["msl"].values[:2][None]),
    },
    static_vars={
        "z": torch.from_numpy(static_vars_ds["z"].values[0]),
        "slt": torch.from_numpy(static_vars_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_vars_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atmos_vars_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atmos_vars_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atmos_vars_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atmos_vars_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atmos_vars_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_vars_ds.latitude.values),
        lon=torch.from_numpy(surf_vars_ds.longitude.values),
        time=(surf_vars_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(level) for level in atmos_vars_ds.pressure_level.values),
    ),
)

print(f"Batch spatial shape: {batch.spatial_shape}")

## Load the Model

In [ ]:
from aurora import Aurora

model = Aurora(use_lora=False)
model.load_checkpoint("microsoft/aurora", "aurora-0.25-pretrained.ckpt")

# Enable activation checkpointing for memory efficiency
model.configure_activation_checkpointing()

model.eval()
model = model.to("cuda")

print("Model loaded successfully!")

## Define the Target

We want to find initial condition perturbations that would cause the temperature at a specific location to be 3K lower than the reference forecast over steps 1-3.

In [ ]:
import numpy as np

# Target location: approximately Paris (48.9°N, 2.3°E)
target_lat = 48.9
target_lon = 2.3

# Find the nearest grid indices
lat_idx = int(torch.argmin(torch.abs(batch.metadata.lat - target_lat)).item())
lon_idx = int(torch.argmin(torch.abs(batch.metadata.lon - target_lon)).item())

actual_lat = batch.metadata.lat[lat_idx].item()
actual_lon = batch.metadata.lon[lon_idx].item()

print(f"Target location: {target_lat}°N, {target_lon}°E (Paris)")
print(f"Nearest grid point: {actual_lat}°N, {actual_lon}°E")
print(f"Grid indices: lat_idx={lat_idx}, lon_idx={lon_idx}")

# Define the target trajectory change: 3K cooler over 4 forecast steps
steps = 4
target_delta = torch.tensor([-3.0, -3.0, -3.0, -3.0])  # Kelvin

print(f"\nTarget: Temperature {target_delta[0].item()}K change over {steps} steps")

## Initialize the Solver

The `InverseRolloutSolver` first computes a reference trajectory, then iteratively optimizes the initial perturbation.

In [ ]:
from aurora import InverseRolloutSolver

# Create the solver
solver = InverseRolloutSolver(
    model=model,
    reference_batch=batch,
    steps=steps,
)

print("Solver initialized.")
print("Computing reference trajectory...")

# Access reference predictions (computed lazily)
ref_preds = solver.reference_predictions
print(f"Reference trajectory computed: {len(ref_preds)} steps")

## Extract Reference Trajectory

Let's look at the reference temperature trajectory at our target point.

In [ ]:
from aurora import extract_timeseries
import matplotlib.pyplot as plt

# Extract reference trajectory
ref_trajectory = extract_timeseries(
    ref_preds,
    var_name="2t",
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    var_type="surf",
)

ref_celsius = ref_trajectory.detach().cpu().numpy() - 273.15
forecast_hours = [6 * (i + 1) for i in range(steps)]

# Compute target trajectory
target_celsius = ref_celsius + target_delta.numpy()

plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, ref_celsius, "b-o", linewidth=2, markersize=8, label="Reference forecast")
plt.plot(forecast_hours, target_celsius, "r--s", linewidth=2, markersize=8, label="Target (reference - 3K)")
plt.fill_between(forecast_hours, ref_celsius, target_celsius, alpha=0.2, color="red")
plt.xlabel("Forecast Lead Time (hours)")
plt.ylabel("Temperature (°C)")
plt.title(f"Reference vs Target Temperature at ({actual_lat}°N, {actual_lon}°E)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Reference trajectory: {ref_celsius} °C")
print(f"Target trajectory:    {target_celsius} °C")

## Run the Optimization

Now we solve for the initial perturbation. We'll restrict the optimization to only perturb the surface temperature field to keep the problem simpler.

In [ ]:
print("Running optimization...")
print("This may take several minutes.\n")

solution = solver.solve(
    target_trajectory_delta=target_delta,
    var_name="2t",
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    var_type="surf",
    # Optimization parameters
    learning_rate=0.1,  # Step size for gradient descent
    max_iterations=50,  # Maximum number of iterations
    tolerance=1e-7,     # Stop if loss change is below this
    regularization=1e-5,  # Penalty on perturbation magnitude
    # Only perturb temperature to keep it simple
    variables_to_perturb={"surf": ["2t"]},
    verbose=True,
)

print(f"\nOptimization complete!")
print(f"Iterations: {solution['iterations']}")
print(f"Converged: {solution['converged']}")
print(f"Final loss: {solution['losses'][-1]:.6f}")

## Analyze the Results

In [ ]:
# Plot the optimization convergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].semilogy(solution["losses"], "b-", linewidth=2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Total Loss")
axes[0].set_title("Total Loss")
axes[0].grid(True, alpha=0.3)

# Trajectory loss (how well we match the target)
axes[1].semilogy(solution["trajectory_losses"], "g-", linewidth=2)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Trajectory Loss")
axes[1].set_title("Trajectory Matching Loss")
axes[1].grid(True, alpha=0.3)

# Regularization loss (perturbation magnitude)
axes[2].plot(solution["reg_losses"], "r-", linewidth=2)
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Regularization Loss")
axes[2].set_title("Regularization Loss")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Achieved vs Target Trajectory

In [ ]:
# Get the achieved trajectory change
achieved_delta = solution["final_trajectory_delta"].cpu().numpy()
target_delta_np = solution["target_trajectory_delta"].cpu().numpy()

# Compute achieved trajectory in Celsius
achieved_celsius = ref_celsius + achieved_delta

plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, ref_celsius, "b-o", linewidth=2, markersize=8, label="Reference forecast")
plt.plot(forecast_hours, target_celsius, "r--s", linewidth=2, markersize=8, label="Target")
plt.plot(forecast_hours, achieved_celsius, "g-^", linewidth=2, markersize=8, label="Achieved (with perturbation)")
plt.xlabel("Forecast Lead Time (hours)")
plt.ylabel("Temperature (°C)")
plt.title(f"Reference vs Target vs Achieved at ({actual_lat}°N, {actual_lon}°E)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Trajectory comparison (in Kelvin change from reference):")
print(f"  Target:   {target_delta_np}")
print(f"  Achieved: {achieved_delta}")
print(f"  Error:    {achieved_delta - target_delta_np}")

## Visualize the Optimized Initial Perturbation

The solver found the temperature perturbation to apply to the initial state. Let's visualize it.

In [ ]:
# Get the optimized perturbation for 2m temperature
temp_perturbation = solution["surf_var_perturbations"]["2t"]
print(f"Perturbation shape: {temp_perturbation.shape}")
print(f"  - Batch: {temp_perturbation.shape[0]}")
print(f"  - History timesteps: {temp_perturbation.shape[1]}")
print(f"  - Spatial: {temp_perturbation.shape[2]} x {temp_perturbation.shape[3]}")

# Plot perturbation for both history timesteps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for t in range(2):
    pert_map = temp_perturbation[0, t].cpu().numpy()
    
    vmax = max(np.abs(pert_map).max(), 0.1)  # Ensure non-zero for colorbar
    
    im = axes[t].imshow(
        pert_map,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[0, 360, -90, 90],
        origin="upper",
    )
    axes[t].plot(actual_lon, actual_lat, "k*", markersize=15, label="Target point")
    axes[t].set_title(f"Temperature perturbation at t-{1-t} (history step {t})")
    axes[t].set_xlabel("Longitude")
    axes[t].set_ylabel("Latitude")
    axes[t].legend()
    plt.colorbar(im, ax=axes[t], label="ΔT (K)")

plt.suptitle("Optimized initial temperature perturbation", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on the region around the target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Define zoom region
lat_min, lat_max = max(-90, actual_lat - 25), min(90, actual_lat + 25)
lon_min, lon_max = max(0, actual_lon - 35), min(360, actual_lon + 35)

lat_mask = (batch.metadata.lat >= lat_min) & (batch.metadata.lat <= lat_max)
lon_mask = (batch.metadata.lon >= lon_min) & (batch.metadata.lon <= lon_max)

lat_indices = torch.where(lat_mask)[0]
lon_indices = torch.where(lon_mask)[0]

for t in range(2):
    pert_zoomed = temp_perturbation[
        0, t, 
        lat_indices[0]:lat_indices[-1]+1, 
        lon_indices[0]:lon_indices[-1]+1
    ].cpu().numpy()
    
    vmax = max(np.abs(pert_zoomed).max(), 0.1)
    
    im = axes[t].imshow(
        pert_zoomed,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[lon_min, lon_max, lat_min, lat_max],
        origin="upper",
    )
    axes[t].plot(actual_lon, actual_lat, "k*", markersize=15, label="Target point")
    axes[t].set_title(f"Zoomed perturbation (history step {t})")
    axes[t].set_xlabel("Longitude")
    axes[t].set_ylabel("Latitude")
    axes[t].legend()
    plt.colorbar(im, ax=axes[t], label="ΔT (K)")

plt.suptitle("Zoomed view of optimized perturbation", fontsize=12)
plt.tight_layout()
plt.show()

# Print statistics
print(f"\nPerturbation statistics:")
print(f"  Min: {temp_perturbation.min().item():.4f} K")
print(f"  Max: {temp_perturbation.max().item():.4f} K")
print(f"  Mean: {temp_perturbation.mean().item():.4f} K")
print(f"  Std: {temp_perturbation.std().item():.4f} K")
print(f"  RMS: {torch.sqrt(torch.mean(temp_perturbation**2)).item():.4f} K")

## Perturbing Multiple Variables

Let's try a more flexible optimization that can perturb multiple variables. This often leads to smaller perturbations overall since the optimizer has more degrees of freedom.

In [ ]:
print("Running optimization with multiple variables...\n")

solution_multi = solver.solve(
    target_trajectory_delta=target_delta,
    var_name="2t",
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    var_type="surf",
    learning_rate=0.1,
    max_iterations=50,
    tolerance=1e-7,
    regularization=1e-5,
    # Perturb temperature, wind, and pressure
    variables_to_perturb={
        "surf": ["2t", "10u", "10v", "msl"],
    },
    verbose=True,
)

print(f"\nOptimization complete!")
print(f"Converged: {solution_multi['converged']}")
print(f"Final loss: {solution_multi['losses'][-1]:.6f}")

In [ ]:
# Compare single-variable vs multi-variable optimization
achieved_single = solution["final_trajectory_delta"].cpu().numpy()
achieved_multi = solution_multi["final_trajectory_delta"].cpu().numpy()

plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, target_delta_np, "r--s", linewidth=2, markersize=8, label="Target")
plt.plot(forecast_hours, achieved_single, "b-o", linewidth=2, markersize=8, label="Single-var (T only)")
plt.plot(forecast_hours, achieved_multi, "g-^", linewidth=2, markersize=8, label="Multi-var (T, u, v, msl)")
plt.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
plt.xlabel("Forecast Lead Time (hours)")
plt.ylabel("Temperature Change (K)")
plt.title("Achieved trajectory change: Single vs Multi-variable optimization")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compare perturbation magnitudes
def compute_rms(perturbations):
    total = 0
    count = 0
    for p in perturbations.values():
        total += (p**2).sum().item()
        count += p.numel()
    return np.sqrt(total / count)

rms_single = compute_rms(solution["surf_var_perturbations"])
rms_multi = compute_rms(solution_multi["surf_var_perturbations"])

print(f"\nPerturbation RMS comparison:")
print(f"  Single-variable: {rms_single:.6f}")
print(f"  Multi-variable:  {rms_multi:.6f}")

## Visualize Multi-Variable Perturbations

In [ ]:
# Plot perturbations for all variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variables = [
    ("2t", "2m Temperature", "K"),
    ("msl", "Mean Sea Level Pressure", "Pa"),
    ("10u", "10m U-Wind", "m/s"),
    ("10v", "10m V-Wind", "m/s"),
]

for idx, (var_name, title, unit) in enumerate(variables):
    ax = axes[idx // 2, idx % 2]
    
    if var_name not in solution_multi["surf_var_perturbations"]:
        ax.text(0.5, 0.5, "Not perturbed", ha="center", va="center")
        ax.set_title(title)
        continue
    
    pert = solution_multi["surf_var_perturbations"][var_name]
    pert_map = pert[0, -1].cpu().numpy()  # Most recent history step
    
    vmax = max(np.abs(pert_map).max(), 1e-10)
    
    im = ax.imshow(
        pert_map,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[0, 360, -90, 90],
        origin="upper",
    )
    ax.plot(actual_lon, actual_lat, "k*", markersize=12)
    ax.set_title(f"{title} perturbation")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label=f"Δ{var_name} ({unit})")

plt.suptitle("Multi-variable optimized perturbations", fontsize=12)
plt.tight_layout()
plt.show()

## Verify the Solution

Let's verify by running a forward rollout with the perturbed initial conditions.

In [ ]:
import dataclasses
from aurora import differentiable_rollout

# Create perturbed batch using the multi-variable solution
perturbed_surf_vars = {}
for k, v in batch.surf_vars.items():
    if k in solution_multi["surf_var_perturbations"]:
        perturbed_surf_vars[k] = v + solution_multi["surf_var_perturbations"][k].cpu()
    else:
        perturbed_surf_vars[k] = v

perturbed_batch = dataclasses.replace(
    batch,
    surf_vars=perturbed_surf_vars,
)

# Run forward pass with perturbed initial conditions
print("Running verification rollout...")
with torch.no_grad():
    perturbed_preds, _ = differentiable_rollout(model, perturbed_batch, steps)

# Extract the trajectory
perturbed_trajectory = extract_timeseries(
    perturbed_preds,
    var_name="2t",
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    var_type="surf",
)

perturbed_celsius = perturbed_trajectory.detach().cpu().numpy() - 273.15
verified_delta = perturbed_celsius - ref_celsius

print("\nVerification results:")
print(f"  Target delta:   {target_delta_np}")
print(f"  Verified delta: {verified_delta}")
print(f"  Error:          {verified_delta - target_delta_np}")

In [ ]:
# Final comparison plot
plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, ref_celsius, "b-o", linewidth=2, markersize=8, label="Reference")
plt.plot(forecast_hours, target_celsius, "r--s", linewidth=2, markersize=8, label="Target")
plt.plot(forecast_hours, perturbed_celsius, "g-^", linewidth=2, markersize=10, label="Verified (with perturbation)")
plt.xlabel("Forecast Lead Time (hours)")
plt.ylabel("Temperature (°C)")
plt.title(f"Final verification at ({actual_lat}°N, {actual_lon}°E)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Cleanup

In [ ]:
model = model.to("cpu")
torch.cuda.empty_cache()
print("Cleanup complete!")

## Summary

In this notebook, we demonstrated how to:

1. **Initialize the InverseRolloutSolver** with a reference forecast
2. **Define a target trajectory change** (e.g., 3K cooler over 4 steps)
3. **Run the optimization** to find initial perturbations
4. **Analyze convergence** through loss plots
5. **Compare single vs multi-variable optimization**
6. **Verify the solution** by running a forward rollout

Key parameters for the solver:
- `learning_rate`: Controls step size (higher = faster but less stable)
- `max_iterations`: Maximum optimization steps
- `regularization`: Penalizes large perturbations (higher = smaller perturbations but potentially worse fit)
- `variables_to_perturb`: Restricts which variables can be modified

This approach is useful for:
- Generating targeted ensemble perturbations
- Understanding model controllability
- Scenario analysis ("what initial change would cause X?")
- Sensitivity studies with specific target changes